# Robustness / convergence sanity check 

In [20]:
import ast
import inspect
import random
from contextlib import nullcontext
from typing import Callable, Dict, List, Tuple
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from transformers import AutoModel, AutoTokenizer

In [21]:
try:
    from IsoScore import IsoScore
except Exception as e:
    raise ImportError("Missing `IsoScore`. Install with: pip install IsoScore") from e

try:
    from dadapy import Data
except Exception as e:
    raise ImportError("Missing `dadapy` (TwoNN/GRIDE). Install with: pip install dadapy") from e

try:
    from skdim.id import MOM, TLE, CorrInt, FisherS, lPCA, MLE, MADA, KNN, ESS
except Exception as e:
    raise ImportError("Missing `scikit-dimension`. Install with: pip install scikit-dimension") from e

In [22]:
RAND_SEED = 42
random.seed(RAND_SEED)
np.random.seed(RAND_SEED)

In [23]:
CSV_PATH = "data/en_ewt-ud-train_sentences.csv"

MODEL_NAME = "bert-base-uncased"
WORD_REP_MODE = "first"   # "first" | "last" | "mean"

EXCLUDE_POS = {"X", "SYM", "PART", "INTJ"}

N_POOL_WORDS = 20_000  
MAX_LEN = 128
BATCH_SIZE_SENT = 16

N_MIN = 200
N_MAX = 10_000
N_POINTS = 18
SEEDS = [0, 1, 2]  

DADAPY_GRID_RANGE_MAX = 64

In [24]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [25]:
def _to_list(x):
    if isinstance(x, list):
        return x
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        try:
            v = ast.literal_eval(s)
            return v if isinstance(v, list) else [v]
        except Exception:
            return s.split()
    return [x]

In [26]:
df_sent = pd.read_csv(CSV_PATH)
if "sentence_id" not in df_sent.columns or "tokens" not in df_sent.columns:
    raise ValueError(f"CSV must contain at least ['sentence_id','tokens'], got {list(df_sent.columns)}")

df_sent["sentence_id"] = df_sent["sentence_id"].astype(str)
df_sent["tokens"] = df_sent["tokens"].apply(_to_list)

has_pos = "pos" in df_sent.columns
if has_pos:
    df_sent["pos"] = df_sent["pos"].apply(_to_list)

rows = []
if has_pos:
    for sid, toks, poss in df_sent[["sentence_id","tokens","pos"]].itertuples(index=False):
        for wid, (tok, p) in enumerate(zip(toks, poss)):
            if p in EXCLUDE_POS:
                continue
            rows.append((sid, wid, tok))
else:
    for sid, toks in df_sent[["sentence_id","tokens"]].itertuples(index=False):
        for wid, tok in enumerate(toks):
            rows.append((sid, wid, tok))

word_df = pd.DataFrame(rows, columns=["sentence_id","word_id","word"])
del rows

In [27]:
print("sentences:", len(df_sent))
print("candidate words:", len(word_df))

N_POOL = min(N_POOL_WORDS, len(word_df))
pool_words = word_df.sample(n=N_POOL, random_state=RAND_SEED, replace=False).reset_index(drop=True)
print("sampled pool:", len(pool_words))

sentences: 10067
candidate words: 189167
sampled pool: 20000


In [28]:
@torch.no_grad()
def embed_words_last_layer(
    df_sentences: pd.DataFrame,
    subset_words: pd.DataFrame,
    model_name: str,
    word_rep_mode: str,
    batch_size: int,
    max_length: int,
    device: str,
) -> Tuple[np.ndarray, np.ndarray]:
    """Return (X_last, filled) where:
      - X_last: (N,D) float32 word vectors
      - filled: (N,) bool mask for successfully aligned words
    """
    if word_rep_mode not in {"first","last","mean"}:
        raise ValueError(f"word_rep_mode must be one of {{first,last,mean}}, got {word_rep_mode}")

    by_sid: Dict[str, List[Tuple[int,int]]] = {}
    for gidx, (sid, wid) in enumerate(subset_words[["sentence_id","word_id"]].itertuples(index=False)):
        by_sid.setdefault(str(sid), []).append((int(gidx), int(wid)))

    sids = list(by_sid.keys())
    df_sel = (
        df_sentences[df_sentences.sentence_id.isin(sids)]
        .drop_duplicates("sentence_id")
        .set_index("sentence_id")
        .loc[sids]
    )

    tokzr = AutoTokenizer.from_pretrained(model_name, use_fast=True, add_prefix_space=True)
    if not getattr(tokzr, "is_fast", False):
        raise ValueError("Fast tokenizer required (use_fast=True) to access enc.word_ids().")

    if tokzr.pad_token is None:
        tokzr.pad_token = tokzr.eos_token

    model = AutoModel.from_pretrained(model_name)
    model.eval()
    model.to(device)

    enc_kwargs = dict(
        is_split_into_words=True,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length,
    )
    if "add_prefix_space" in inspect.signature(tokzr.__call__).parameters:
        enc_kwargs["add_prefix_space"] = True

    amp_ctx = torch.cuda.amp.autocast if device == "cuda" else nullcontext

    N = len(subset_words)
    X_last = None
    filled = np.zeros(N, dtype=bool)

    for start in tqdm(range(0, len(sids), batch_size), desc=f"Embedding pool ({model_name})"):
        batch_ids = sids[start:start+batch_size]
        batch_tokens = df_sel.loc[batch_ids, "tokens"].tolist()

        enc = tokzr(batch_tokens, **enc_kwargs)
        word_ids_list = [enc.word_ids(batch_index=bi) for bi in range(len(batch_ids))]
        enc_t = {k: v.to(device) for k, v in enc.items()}

        with amp_ctx():
            out = model(**enc_t)
            h = out.last_hidden_state.detach().cpu().numpy().astype(np.float32)  # (B,T,D)

        if X_last is None:
            X_last = np.zeros((N, h.shape[-1]), dtype=np.float32)

        for b, sid in enumerate(batch_ids):
            mp: Dict[int, List[int]] = {}
            for tidx, wid in enumerate(word_ids_list[b]):
                if wid is not None:
                    mp.setdefault(int(wid), []).append(int(tidx))

            for gidx, wid in by_sid.get(sid, []):
                toks = mp.get(wid)
                if not toks:
                    continue

                if word_rep_mode == "first":
                    vec = h[b, toks[0], :]
                elif word_rep_mode == "last":
                    vec = h[b, toks[-1], :]
                else:
                    vec = h[b, toks, :].mean(axis=0)

                X_last[gidx] = vec
                filled[gidx] = True

        del enc, enc_t, out, h
        if device == "cuda":
            torch.cuda.empty_cache()

    assert X_last is not None
    missing = int((~filled).sum())
    if missing:
        print(f"Missing vectors: {missing} / {N} (usually truncation at MAX_LEN).")

    del model
    if device == "cuda":
        torch.cuda.empty_cache()

    return X_last, filled

In [29]:
X_pool_raw, filled = embed_words_last_layer(
    df_sentences=df_sent,
    subset_words=pool_words,
    model_name=MODEL_NAME,
    word_rep_mode=WORD_REP_MODE,
    batch_size=BATCH_SIZE_SENT,
    max_length=MAX_LEN,
    device=device,
)

X_pool = X_pool_raw[filled]
print("X_pool:", X_pool.shape)


Embedding pool (bert-base-uncased):   0%|          | 0/504 [00:00<?, ?it/s]

Missing vectors: 4 / 20000 (usually truncation at MAX_LEN).
X_pool: (19996, 768)


In [30]:
EPS = 1e-9

def _center(X: np.ndarray) -> np.ndarray:
    return X - X.mean(0, keepdims=True)

def _eigvals_from_X(X: np.ndarray) -> np.ndarray:
    """Eigenvalues of covariance up to a constant via SVD of centered X (descending)."""
    Xc = _center(X.astype(np.float32, copy=False))
    try:
        _, S, _ = np.linalg.svd(Xc, full_matrices=False)
        lam = (S**2).astype(np.float64)
        lam.sort()
        return lam[::-1]
    except Exception:
        return np.array([], dtype=np.float64)

def _jitter_unique(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """Add tiny noise if there are duplicate rows (helps NN-based estimators)."""
    try:
        if np.unique(X, axis=0).shape[0] < X.shape[0]:
            rng = np.random.default_rng(RAND_SEED)
            X = X + rng.normal(scale=eps, size=X.shape).astype(X.dtype)
    except Exception:
        pass
    return X

# ---------- Isotropy ----------
def IsoScore_once(X: np.ndarray) -> float:
    return float(IsoScore.IsoScore(X))

def SpectralFlatness_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return float("nan")
    gm = np.exp(np.mean(np.log(lam + EPS)))
    am = float(lam.mean() + EPS)
    return float(gm / am)

def vMF_kappa_once(X: np.ndarray) -> float:
    if X.shape[0] < 2:
        return float("nan")
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + EPS)
    R = float(np.linalg.norm(Xn.mean(axis=0)))
    d = Xn.shape[1]
    if R < EPS:
        return 0.0
    # standard closed-form approximation (Sra 2012-style)
    return float(max(R * (d - R**2) / (1.0 - R**2 + EPS), 0.0))

# ---------- Linear ID (spectrum-based) ----------
def PCA99_once(X: np.ndarray, var_ratio: float = 0.99) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return float("nan")
    c = np.cumsum(lam)
    thr = c[-1] * var_ratio
    return float(np.searchsorted(c, thr) + 1)

def EffectiveRank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return float("nan")
    p = lam / (lam.sum() + EPS)
    H = -(p * np.log(p + EPS)).sum()
    return float(np.exp(H))

def ParticipationRatio_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return float("nan")
    s1 = lam.sum()
    s2 = (lam**2).sum()
    return float((s1**2) / (s2 + EPS))

def StableRank_once(X: np.ndarray) -> float:
    lam = _eigvals_from_X(X)
    if lam.size == 0:
        return float("nan")
    return float(lam.sum() / (lam.max() + EPS))

def lPCA_FO_once(X: np.ndarray) -> float:
    est = lPCA(ver="FO")
    est.fit(_jitter_unique(X))
    return float(getattr(est, "dimension_", np.nan))

def lPCA99_once(X: np.ndarray) -> float:
    est = lPCA(ver="ratio", alphaRatio=0.99)
    est.fit(_jitter_unique(X))
    return float(getattr(est, "dimension_", np.nan))

# ---------- Nonlinear ID ----------
def TwoNN_once(X: np.ndarray) -> float:
    d = Data(coordinates=_jitter_unique(X))
    id_est, _, _ = d.compute_id_2NN()
    return float(id_est)

def GRIDE_once(X: np.ndarray) -> float:
    d = Data(coordinates=_jitter_unique(X))
    d.compute_distances(maxk=DADAPY_GRID_RANGE_MAX)
    ids, _, _ = d.return_id_scaling_gride(range_max=DADAPY_GRID_RANGE_MAX)
    return float(ids[-1])

def _skdim_once(cls) -> Callable[[np.ndarray], float]:
    def _fn(X: np.ndarray) -> float:
        est = cls()
        est.fit(_jitter_unique(X))
        return float(getattr(est, "dimension_", np.nan))
    return _fn

# skdim estimators
MOM_once     = _skdim_once(MOM)
TLE_once     = _skdim_once(TLE)
CorrInt_once = _skdim_once(CorrInt)
FisherS_once = _skdim_once(FisherS)
MLE_once     = _skdim_once(MLE)
MADA_once    = _skdim_once(MADA)
KNN_once     = _skdim_once(KNN)
ESS_once     = _skdim_once(ESS)

In [31]:
ISO_METRICS: Dict[str, Callable[[np.ndarray], float]] = {
    "IsoScore": IsoScore_once,
    "Spectral Flatness": SpectralFlatness_once,
    "vMF κ": vMF_kappa_once,
}

LINEAR_ID_METRICS: Dict[str, Callable[[np.ndarray], float]] = {
    "PCA@99": PCA99_once,
    "Effective Rank": EffectiveRank_once,
    "Participation Ratio": ParticipationRatio_once,
    "Stable Rank": StableRank_once,
    "lPCA FO": lPCA_FO_once,
    "lPCA@0.99": lPCA99_once,
}

NONLINEAR_ID_METRICS: Dict[str, Callable[[np.ndarray], float]] = {
    "TwoNN": TwoNN_once,
    "GRIDE": GRIDE_once,
    "MLE": MLE_once,
    "MOM": MOM_once,
    "TLE": TLE_once,
    "CorrInt": CorrInt_once,
    "FisherS": FisherS_once,
    "MADA": MADA_once,
    "ESS": ESS_once,
    "KNN": KNN_once,
}


In [32]:

ALL_METRICS: Dict[str, Callable[[np.ndarray], float]] = {}
ALL_METRICS.update(ISO_METRICS)
ALL_METRICS.update(LINEAR_ID_METRICS)
ALL_METRICS.update(NONLINEAR_ID_METRICS)

print('ISO metrics:', list(ISO_METRICS))
print('Linear ID metrics:', list(LINEAR_ID_METRICS))
print('Nonlinear ID metrics:', list(NONLINEAR_ID_METRICS))

ISO metrics: ['IsoScore', 'Spectral Flatness', 'vMF κ']
Linear ID metrics: ['PCA@99', 'Effective Rank', 'Participation Ratio', 'Stable Rank', 'lPCA FO', 'lPCA@0.99']
Nonlinear ID metrics: ['TwoNN', 'GRIDE', 'MLE', 'MOM', 'TLE', 'CorrInt', 'FisherS', 'MADA', 'ESS', 'KNN']


In [33]:
def _n_grid(n_min: int, n_max: int, n_points: int) -> np.ndarray:
    n_min = int(n_min)
    n_max = int(n_max)
    if n_min < 2:
        n_min = 2
    if n_max < n_min:
        n_max = n_min
    grid = np.logspace(np.log10(n_min), np.log10(n_max), n_points)
    grid = np.unique(np.round(grid).astype(int))
    return grid

def _sample_idx(N: int, n: int, seed: int) -> np.ndarray:
    ss = np.random.SeedSequence([int(RAND_SEED), int(seed), int(n)])
    rng = np.random.default_rng(ss)
    return rng.choice(N, size=n, replace=False)

def safe_call(fn: Callable[[np.ndarray], float], X: np.ndarray) -> float:
    try:
        return float(fn(X))
    except Exception:
        return float("nan")

In [ ]:
N_pool = X_pool.shape[0]
n_max_eff = min(N_MAX, N_pool)
grid = _n_grid(N_MIN, n_max_eff, N_POINTS)
print("N grid:", grid[:5], "...", grid[-5:], f"(len={len(grid)})")
print("Pool size:", N_pool)

rows = []
for n in tqdm(grid, desc="Convergence sweep (N)"):
    vals_by_metric = {m: [] for m in ALL_METRICS}
    for seed in SEEDS:
        idx = _sample_idx(N_pool, int(n), int(seed))
        Xn = X_pool[idx].astype(np.float32, copy=False)

        for m, fn in ALL_METRICS.items():
            vals_by_metric[m].append(safe_call(fn, Xn))

    for m, vals in vals_by_metric.items():
        vals = np.asarray(vals, dtype=float)
        rows.append(
            dict(
                metric=m,
                N=int(n),
                mean=float(np.nanmean(vals)),
                std=float(np.nanstd(vals, ddof=1)),
            )
        )

In [35]:
df_conv = pd.DataFrame(rows)
df_iso = df_conv[df_conv.metric.isin(ISO_METRICS.keys())].copy()
df_lin = df_conv[df_conv.metric.isin(LINEAR_ID_METRICS.keys())].copy()
df_nl  = df_conv[df_conv.metric.isin(NONLINEAR_ID_METRICS.keys())].copy()

df_conv.head()

,metric,N,mean,std
0,IsoScore,200,0.072406,0.015987
1,Spectral Flatness,200,0.504305,0.027677
2,vMF κ,200,504.906349,9.565610
3,PCA@99,200,178.666667,3.214550
4,Effective Rank,200,110.580725,7.385827


In [ ]:
def plot_convergence(ax, df, metric_order: List[str], ylabel: str, title: str):
    for metric in metric_order:
        g = df[df.metric == metric].sort_values("N")
        if g.empty:
            continue
        x = g["N"].to_numpy(dtype=float)
        y = g["mean"].to_numpy(dtype=float)
        s = g["std"].to_numpy(dtype=float)
        ax.plot(x, y, label=metric)
        ax.fill_between(x, y - s, y + s, alpha=0.15)

    ax.set_xscale("log")
    ax.set_xlabel("N samples (log scale)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.25)
    ax.legend(frameon=False, fontsize=8)

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(8, 9), sharex=True)

In [41]:
plot_convergence(
    axes[0],
    df_iso,
    list(ISO_METRICS.keys()),
    ylabel="estimate",
    title="Isotropy (convergence)",
)

plot_convergence(
    axes[1],
    df_lin,
    list(LINEAR_ID_METRICS.keys()),
    ylabel="ID estimate",
    title="Linear intrinsic dimensionality (convergence)",
)

plot_convergence(
    axes[2],
    df_nl,
    list(NONLINEAR_ID_METRICS.keys()),
    ylabel="ID estimate",
    title="Nonlinear intrinsic dimensionality (convergence)",
)

fig.tight_layout()



plt.show()
